File and Data Field Descriptions
**train.csv** - Personal records for about two-thirds (~8700) of the passengers, to be used as training data.
**PassengerId** - A unique Id for each passenger. Each Id takes the form gggg_pp where gggg indicates a group the passenger is travelling with and pp is their number within the group. People in a group are often family members, but not always.
**HomePlanet** - The planet the passenger departed from, typically their planet of permanent residence.
**CryoSleep** - Indicates whether the passenger elected to be put into suspended animation for the duration of the voyage. Passengers in cryosleep are confined to their cabins.
**Cabin** - The cabin number where the passenger is staying. Takes the form deck/num/side, where side can be either P for Port or S for Starboard.
**Destination** - The planet the passenger will be debarking to.
**Age** - The age of the passenger.
**VIP** - Whether the passenger has paid for special VIP service during the voyage.
**RoomService, FoodCourt, ShoppingMall, Spa, VRDeck** - Amount the passenger has billed at each of the Spaceship Titanic's many luxury amenities.
**Name** - The first and last names of the passenger.
**Transported** - Whether the passenger was transported to another dimension. This is the target, the column you are trying to predict.
**test.csv** - Personal records for the remaining one-third (~4300) of the passengers, to be used as test data. Your task is to predict the value of Transported for the passengers in this set.
**sample_submission.csv** - A submission file in the correct format.
**PassengerId** - Id for each passenger in the test set.
**Transported** - The target. For each passenger, predict either True or False.

In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns


In [27]:
train_data =pd.read_csv('train.csv')
train_data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [28]:
test_data = pd.read_csv('test.csv')
test_data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


In [29]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [30]:
train_data['HomePlanet'].value_counts()

HomePlanet
Earth     4602
Europa    2131
Mars      1759
Name: count, dtype: int64

In [31]:
train_data['CryoSleep'].value_counts()

CryoSleep
False    5439
True     3037
Name: count, dtype: int64

In [32]:
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_cols:
    train_data[col].fillna(train_data[col].median(), inplace=True)

C:\Users\T14\AppData\Local\Temp\ipykernel_16820\378914634.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_data[col].fillna(train_data[col].median(), inplace=True)


In [33]:
# For categorical columns
cat_cols = ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP', 'Name']
for col in cat_cols:
         train_data[col].fillna(train_data[col].mode()[0], inplace=True)
   
print('Missing values after imputation:')
train_data.isnull().sum()
   

Missing values after imputation:


C:\Users\T14\AppData\Local\Temp\ipykernel_16820\2498856416.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_data[col].fillna(train_data[col].mode()[0], inplace=True)
C:\Users\T14\AppData\Local\Temp\ipykernel_16820\2498856416.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_data[col].fillna(tra

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

In [34]:
train_data['CryoSleep'].value_counts()

CryoSleep
False    5656
True     3037
Name: count, dtype: int64

In [35]:
# Feature Engineering
# # Split Cabin into Deck, Num, Side
train_data[['Deck', 'Num', 'Side']] = train_data['Cabin'].str.split('/', expand=True)
train_data['Num'] = pd.to_numeric(train_data['Num'], errors='coerce')
train_data.drop('Cabin', axis=1, inplace=True)
# Extract group from PassengerId
train_data['Group'] = train_data['PassengerId'].str.split('_').str[0].astype(int)
train_data['GroupSize'] = train_data.groupby('Group')['Group'].transform('count')
# Drop Name and PassengerId as they might not be useful
train_data.drop(['Name', 'PassengerId'], axis=1, inplace=True)
train_data.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,Num,Side,Group,GroupSize
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False,B,0,P,1,1
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True,F,0,S,2,1
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,A,0,S,3,2
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,A,0,S,3,2
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True,F,1,S,4,1


In [36]:
# Encode categorical variables
from sklearn.calibration import LabelEncoder


le = LabelEncoder()
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
for col in cat_cols:
    train_data[col] = le.fit_transform(train_data[col])

# Convert boolean to int
train_data['Transported'] = train_data['Transported'].astype(int)
train_data.head()
   

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,Num,Side,Group,GroupSize
0,1,0,2,39.0,0,0.0,0.0,0.0,0.0,0.0,0,1,0,0,1,1
1,0,0,2,24.0,0,109.0,9.0,25.0,549.0,44.0,1,5,0,1,2,1
2,1,0,2,58.0,1,43.0,3576.0,0.0,6715.0,49.0,0,0,0,1,3,2
3,1,0,2,33.0,0,0.0,1283.0,371.0,3329.0,193.0,0,0,0,1,3,2
4,0,0,2,16.0,0,303.0,70.0,151.0,565.0,2.0,1,5,1,1,4,1


In [37]:
# Scale numerical features\n",
from sklearn.discriminant_analysis import StandardScaler


scaler = StandardScaler()
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Num', 'Group', 'GroupSize']
train_data[num_cols] = scaler.fit_transform(train_data[num_cols])
train_data.head()



,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,Num,Side,Group,GroupSize
0,1,0,2,0.711945,0,-0.333105,-0.281027,-0.283579,-0.270626,-0.263003,0,1,-1.191744,0,-1.734409,-0.648735
1,0,0,2,-0.334037,0,-0.168073,-0.275387,-0.241771,0.217158,-0.224205,1,5,-1.191744,1,-1.734034,-0.648735
2,1,0,2,2.036857,1,-0.268001,1.959998,-0.283579,5.695623,-0.219796,0,0,-1.191744,1,-1.733660,-0.022268
3,1,0,2,0.293552,0,-0.333105,0.523010,0.336851,2.687176,-0.092818,0,0,-1.191744,1,-1.733660,-0.022268
4,0,0,2,-0.891895,0,0.125652,-0.237159,-0.031059,0.231374,-0.261240,1,5,-1.189769,1,-1.733286,-0.648735


In [38]:
# Save the preprocessed data
train_data.to_csv('train_preprocessed.csv', index=False)
print('Preprocessed data saved to train_preprocessed.csv')

Preprocessed data saved to train_preprocessed.csv


In [39]:
features = train_data.drop('Transported', axis=1)
target = train_data['Transported']
X_train, X_val, y_train, y_val = train_test_split(features, target, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
print(classification_report(y_val, y_pred))


              precision    recall  f1-score   support

           0       0.78      0.80      0.79       861
           1       0.80      0.78      0.79       878

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739



In [45]:
# 1. Load the actual Test Data
test_data = pd.read_csv('test.csv')
passenger_ids = test_data['PassengerId'] # Save this for the final file!

# 2. Apply the SAME missing value logic
for col in num_cols:
    # Use the median from TRAIN to fill TEST (to avoid data leakage)
    test_data[col].fillna(train_data[col].median(), inplace=True)

for col in cat_cols:
    test_data[col].fillna(train_data[col].mode()[0], inplace=True)

# 3. Feature Engineering (Repeat your Cabin split)
# Always create all three columns, even if some values are missing
cabin_split = test_data['Cabin'].str.split('/', expand=True)

# Ensure all three columns exist
for i in range(3 - cabin_split.shape[1]):
    cabin_split[cabin_split.shape[1]] = np.nan
cabin_split.columns = ['Deck', 'Num', 'Side']
test_data[['Deck', 'Num', 'Side']] = cabin_split

# Convert Num to numeric and fill missing values
test_data['Num'] = pd.to_numeric(test_data['Num'], errors='coerce')
test_data['Deck'] = test_data['Deck'].fillna('Unknown')
test_data['Side'] = test_data['Side'].fillna('Unknown')
test_data['Group'] = test_data['PassengerId'].str.split('_').str[0].astype(int)
test_data['GroupSize'] = test_data.groupby('Group')['Group'].transform('count')

# 4. Drop the same columns
test_data.drop(['Cabin', 'Name', 'PassengerId'], axis=1, inplace=True)

# 5. Encoding & Scaling
# IMPORTANT: Use the scaler you already .fit() on the training data
test_data[num_cols] = scaler.transform(test_data[num_cols]) 



C:\Users\T14\AppData\Local\Temp\ipykernel_16820\2513202553.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_data[col].fillna(train_data[col].median(), inplace=True)


KeyError: 'Num'

In [ ]:
predictions = model.predict(test_data)

In [ ]:
import pandas as pd

# 1. Make predictions (this usually returns 0/1 or True/False)
predictions = model.predict(X_test)

# 2. Create the submission DataFrame
# Note: Ensure 'predictions' is converted to boolean (True/False) if it's currently 0/1
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Transported": predictions.astype(bool)
})

# 3. Save to CSV
submission.to_csv("submission.csv", index=False)